In [2]:
# visualize HLS rgb
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display

In [3]:
# BASE_DIR = "/home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/CCM-12TYP-2024/2026/06/16/16/21/43/075766/12TYP/_hls_downloads/"  #"/home/jovyan/my-public-bucket/HLS_phenometrics/"  
BASE_DIR = "/home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/Test-03WWM-2019/2026/07/29/19/42/16/577549/03WWM/_hls_downloads/"
MGRS_TILE = "03WWM"               
START_DATE = "2019-01-01"
END_DATE = "2019-03-01"

In [4]:
def parse_date(date_str):
    """Parses standard YYYY-MM-DD date strings."""
    return datetime.strptime(date_str, "%Y-%m-%d")

def normalize_band(band_data, p_low=2, p_high=98):
    """Normalizes the band data to 0-1 for RGB plotting using percentiles to remove outliers."""
    # Mask out no-data values (HLS uses -9999 for fill)
    valid_data = band_data[band_data > -9999]
    if len(valid_data) == 0:
        return np.zeros_like(band_data, dtype=np.float32)
        
    vmin = np.percentile(valid_data, p_low)
    vmax = np.percentile(valid_data, p_high)
    
    # Avoid division by zero if an image is completely flat
    if vmax == vmin:
        return np.zeros_like(band_data, dtype=np.float32)
        
    normalized = np.clip((band_data - vmin) / (vmax - vmin), 0, 1)
    
    # Set original no-data areas to 0 (black/transparent)
    normalized[band_data <= -9999] = 0
    return normalized

def load_rgb(band_paths):
    """Loads and stacks the RGB bands."""
    with rasterio.open(band_paths['R']) as src:
        red = src.read(1)
    with rasterio.open(band_paths['G']) as src:
        green = src.read(1)
    with rasterio.open(band_paths['B']) as src:
        blue = src.read(1)
        
    r_norm = normalize_band(red)
    g_norm = normalize_band(green)
    b_norm = normalize_band(blue)
    
    return np.dstack((r_norm, g_norm, b_norm))

In [7]:
start_dt = parse_date(START_DATE)
end_dt = parse_date(END_DATE)

available_scenes = {}

# Iterate through the folders in the base directory
for folder_name in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder_name)
    
    # Check if it's a directory and matches the HLS folder naming convention
    if not os.path.isdir(folder_path) or not folder_name.startswith("HLS."):
        continue
    print(folder_path)
    parts = folder_name.split('.')
    # Expected format: HLS . L30/S30 . T12TYP . 2023006T175505 . v2 . 0
    if len(parts) < 4:
        continue
        
    product = parts[1]               # e.g., L30 or S30
    tile = parts[2]                  # e.g., T12TYP
    date_time_str = parts[3]         # e.g., 2023006T175505
    
    # Check if this folder matches our target tile (accounting for 'T' prefix)
    if tile != f"T{MGRS_TILE}":
        continue
        
    # Extract the Julian date (YYYYJJJ)
    julian_date_str = date_time_str[:7]
    try:
        scene_date = datetime.strptime(julian_date_str, "%Y%j")
    except ValueError:
        continue
        
    if start_dt <= scene_date <= end_dt:
        red_file = os.path.join(folder_path, f"{folder_name}.B04.tif")
        green_file = os.path.join(folder_path, f"{folder_name}.B03.tif")
        blue_file = os.path.join(folder_path, f"{folder_name}.B02.tif")
        
        if os.path.exists(red_file) and os.path.exists(green_file) and os.path.exists(blue_file):
            date_label = scene_date.strftime("%Y-%m-%d")
            # Include time in key to prevent overwriting if S30 and L30 have scenes on the same day
            time_label = date_time_str[8:12] # Get HHMM for uniqueness
            key = f"{date_label} {time_label} ({product})"
            
            available_scenes[key] = {
                "R": red_file,
                "G": green_file,
                "B": blue_file
            }

sorted_scenes = dict(sorted(available_scenes.items()))

if not sorted_scenes:
    print(f"No valid RGB data found for tile {MGRS_TILE} between {START_DATE} and {END_DATE}.")
else:
    print(f"Found {len(sorted_scenes)} scenes.")

OSError: [Errno 107] Transport endpoint is not connected: '/home/jovyan/my-private-bucket/dps_output/hls-phenometrics/main/CCM-12TYP-2024/2026/06/16/16/21/43/075766/12TYP/_hls_downloads/'

In [ ]:
sorted_scenes_temp = sorted_scenes # dict(list(sorted_scenes.items())[:5])

In [6]:
def plot_scene(scene_key):
    print(f"Loading scene: {scene_key}...")
    band_paths = sorted_scenes_temp[scene_key]
    
    rgb_image = load_rgb(band_paths)
    
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_image)
    plt.title(f"HLS Tile: {MGRS_TILE} | {scene_key}")
    plt.axis('off')
    plt.show()

# Create interactive slider
if sorted_scenes_temp:
    scene_selector = widgets.SelectionSlider(
        options=list(sorted_scenes_temp.keys()),
        description='Date:',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        layout=widgets.Layout(width='80%')
    )
    
    widgets.interact(plot_scene, scene_key=scene_selector)

NameError: name 'sorted_scenes_temp' is not defined